# Owner-authorized B1 exploratory GPU pilot — execution v1
Owner: Vinuthan T; explicit permission recorded 2026-10-15. Run cells in order on a fresh free Colab GPU runtime (T4 if available); no Drive mount. Do not purchase compute automatically.

Reuses unchanged, CPU-verified71/71 code. First upload the small `vision_b1_pilot_code_v1.zip`; later upload only the pinned development ZIP when asked. No full71-case rerun: GPU dependency and synthetic forward/optimizer/Grad-CAM checks are performed before dental upload.

Protocol:680Training/53Validation, MGI1–4 four outputs, ResNet50 V2, head-only/frozenBN, seed42, batch16,lr0.001,max5epochs,patience2. Early stopping may finish before5epochs. The181holds/NP015/protected/Test stay excluded. Three fixed Validation CAMs only. No automatic retry if a check fails.

**Provisional labels; non-clinical; non-final; participant independence unverified.** No MGI0/healthy detection; Validation MGI1 support is1. Display previews may clip colours; model arrays remain unclipped. Original five-grade research requirements stay unchanged.


In [ ]:
# Upload ONLY vision_b1_pilot_code_v1.zip. No dental data in this step.
from google.colab import files
import hashlib, json, pathlib, tempfile, zipfile, stat, sys, os
uploaded = files.upload()
assert set(uploaded) == {"vision_b1_pilot_code_v1.zip"}, "Upload the code-only ZIP only"
raw = uploaded["vision_b1_pilot_code_v1.zip"]
assert hashlib.sha256(raw).hexdigest() == "eba79c8f5539edf5014546986466157688f261397c05d9a63437ef768c40de30", "Wrong code ZIP"
import io
archive = zipfile.ZipFile(io.BytesIO(raw))
manifest = json.loads(archive.read("verification_manifest.json"))
EXPECTED_FILES = {"src/vision/b1_development_bundle_v1.py": "7ad20bcafce99797164239275954809cc11a7750922d6998443387aa91c7119c", "src/vision/b1_jpeg_source_v1.py": "5f28c0deb73f74a10afb8720300866877602e1c012ddecec092d46d23a1deefb", "src/vision/colour_b1_iccv2_generated.py": "6c611e1c9d77e7af1122d835cc7ce4d17699cbbb8497a6c279b32cd81a58c45c", "src/vision/colour_b1_prototype.py": "d3108ea3bf7f310f1b46960174d4936299ece19d762f4ca1ee9d900572b90856", "src/vision/colour_b1_streaming.py": "5b3a6bc893d1f6fc451c74e69f4865de00565b91bd19dc935baee6476cc5ec67", "src/vision/colour_policy.py": "2bba567545143687f49b6e3ae63fdd72d1eacc10977d5ec373f00a4b5451a97b", "src/vision/colour_qa.py": "4ad2b81c715b01e624b527d1971cc303740ddf559461f2d953c831956c1e82e3", "src/vision/exploratory_bundle.py": "dda776ca1558ed4da795df7cee47c0fe19d382cf32c37ca89d22f4141974b615", "src/vision/exploratory_bundle_b1.py": "01b20dc64f82c9d6d4108ba847f28073bdd7fe8fc4ccd4255119a2fe219d896b", "src/vision/exploratory_bundle_v4.py": "c914cc05ad2be4ff08873a5d2270b5836e0f84219a539435beeca09b57ad76cd", "src/vision/exploratory_pilot.py": "c5bf6cba173390f914a44c4537b91d4f2a8e150aa0f5e4842c2ca94bb1a4d535", "src/vision/exploratory_pilot_b1_v1.py": "876ec09dd7e9766bbc847416398e78d1ce320fa38a20bbaa952d35750eb5be05", "src/vision/gamut_mapping_feasibility.py": "41cd1d49fcebbb08d45fa1951755d4141a4611630ed9b081f26e5a9b273f368a", "src/vision/model_pipeline.py": "c11ad18912575f3567d0baf0eefb1bfbb5eb00fbfa075880fe84aef1ccd34b3c", "src/vision/train_pipeline.py": "e7a979b0875af0e459496bca79ce3c80e594acf2639ea7dc9d8659ec75ffc695", "src/vision/verify_b1_pilot_runtime_v1.py": "2bb4b58567db50df72eefa7535d3c739e36bd8c214b4e5199899c3c9c6267538", "tests/test_b1_local_export_v1.py": "e91ced7446b64eadcb2a4f4d1ca5282e04d8c842033a627559400b488977906d", "tests/test_colour_b1_iccv2_generated.py": "fb110d8f7dc27bef91e055e5ef257ca7f05d4a895822b223e66763096421e9d8", "tests/test_exploratory_bundle_b1.py": "a577869587a3eb516ead83ba2f1fa6832a699781a91a406c2325b710dc596068", "tests/test_exploratory_pilot_b1_v1.py": "e666a5ae9f7c0a45ce9abca649981455df2992d2e01b78be25ccdf718d84dfc3"}
assert manifest["files"] == EXPECTED_FILES
assert len(archive.namelist()) == len(EXPECTED_FILES)+1
assert set(archive.namelist()) == set(EXPECTED_FILES)|{"verification_manifest.json"}
root = pathlib.Path(tempfile.mkdtemp(prefix="b1_pilot_generated_", dir="/content"))
for info in archive.infolist():
    name=info.filename; p=pathlib.PurePosixPath(name)
    assert not p.is_absolute() and ".." not in p.parts and "\\" not in name
    assert not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16) and info.file_size <= 1048576
    data=archive.read(name)
    if name in EXPECTED_FILES: assert hashlib.sha256(data).hexdigest()==EXPECTED_FILES[name]
    target=root/name; target.parent.mkdir(parents=True,exist_ok=True); target.write_bytes(data)
sys.path.insert(0,str(root)); os.chdir(root)
os.environ["PYTHONDONTWRITEBYTECODE"]="1"; sys.dont_write_bytecode=True
print("Fresh code extraction and",len(EXPECTED_FILES),"source hashes verified:",root)


In [ ]:
# Check actual GPU compatibility before any dental upload or weight download.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import torch, torchvision, numpy as np, PIL, platform, shutil, gc
from src.vision import exploratory_pilot_b1_v1 as pilot
from src.vision import exploratory_pilot as weights_source
from src.vision import train_pipeline as trainer
from src.vision import colour_b1_prototype as b1
from src.vision import exploratory_bundle as legacy
assert not pathlib.Path('/content/drive').exists(), 'STOP: isolated runtime without Drive required'
assert torch.cuda.is_available(), 'STOP: Runtime > Change runtime type > T4 GPU, then reconnect'
assert shutil.disk_usage('/content').free >= 8*1024**3, 'STOP: need at least8GiB free disk'
torchvision.ops.nms(torch.tensor([[0.,0.,1.,1.]]),torch.tensor([1.]),.5)
trainer.seed_everything(42)
network=pilot.make_model(pretrained=False)
model=network['model'].to('cuda'); config=pilot.configuration()
trainer.configure_trainable_layers(model,config); trainer.training_mode(model,config)
old_head=model.fc.weight.detach().clone(); old_bn=model.bn1.running_mean.detach().clone()
values=np.zeros((224,224,3),dtype=np.float32); values[0,0]=[-.1,1.1,.5]
prepared=b1.B1Input(np.zeros((224,224,3),dtype=np.uint8),b1.normalize(values),(0,0,224,224),{'display_only':True})
optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=.001)
logits=model(torch.tensor(np.stack([prepared.normalized_chw]*2),device='cuda'))
assert logits.shape==(2,4) and torch.isfinite(logits).all()
loss=torch.nn.functional.cross_entropy(logits,torch.tensor([0,3],device='cuda'))
loss.backward(); optimizer.step()
assert not torch.equal(old_head,model.fc.weight)
assert torch.equal(old_bn,model.bn1.running_mean)
assert all(not p.requires_grad for n,p in model.named_parameters() if not n.startswith('fc.'))
hooks=len(network['cam_layer']._forward_hooks)
views=pilot.gradcam(model,network['cam_layer'],prepared)
assert views['overlay'].shape==(224,224,3) and len(network['cam_layer']._forward_hooks)==hooks
GPU_PREFLIGHT={'status':'PASS','scope':'generated-only GPU compatibility checks',
 'python':platform.python_version(),'torch':str(torch.__version__),'torchvision':str(torchvision.__version__),
 'numpy':np.__version__,'Pillow':PIL.__version__,'gpu':torch.cuda.get_device_name(),
 'gpu_total_bytes':torch.cuda.get_device_properties(0).total_memory,
 'synthetic_checks':['four_logit_forward','head_optimizer_step','frozen_backbone_and_bn','gradcam_alignment_and_hook_cleanup'],
 'dental_training_performed':False}
print(json.dumps(GPU_PREFLIGHT,indent=2))
del model,network,optimizer,logits,loss,views,old_head,old_bn,prepared,values
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# Record the actual owner permission; no clinical/final approval is asserted.
OWNER_RECORD = {'status': 'owner_confirmed', 'authorizer': 'Vinuthan T (project owner)', 'date': '2026-10-15', 'scope': 'upload_exploratory_b1_development_v1', 'evidence_reference': 'Owner explicit YES in this Codex conversation on 2026-10-15: verified 733-image bundle upload, official ResNet50 V2 download, max5epoch MGI1–4 head-only GPU pilot, Validation and three Grad-CAM outputs;181holds/Test excluded.', 'archive_sha256': 'a8a4f7f339c589f81335d7afb663f4cf62d30e217f65bc480683d0a770afed70', 'bundle_manifest_sha256': '8e7ff195009a5e781c216795fd2a943db17a0b3ec8fd1c84760f3bf9878a4c9f', 'code_zip_sha256': 'eba79c8f5539edf5014546986466157688f261397c05d9a63437ef768c40de30', 'verified_cpu_runtime_evidence_sha256': '8627aff3e7dfd744924aaa855661bf016e35043d407bdee99066bbb55384d11e', 'cloud_upload_authorized': True, 'official_resnet50_v2_download_authorized': True, 'training_authorized': True, 'max_epochs': 5, 'final_research_approved': False, 'clinical_labels_approved': False, 'participant_independence_verified': False, 'acknowledgements': {'provisional_labels': True, 'non_clinical_non_final': True, 'participant_independence_unverified': True, 'no_test_access': True, 'missing_mgi0_scarce_mgi1': True, 'no_final_approval_or_promotion': True}}
CLOUD_DATA_AUTHORIZATION=OWNER_RECORD
EXPECTED_DATA_ZIP="pp1_b1_dev_v1_20261015T085436527238Z.zip"
EXPECTED_DATA_SHA256=OWNER_RECORD['archive_sha256']
EXPECTED_MANIFEST_SHA256=OWNER_RECORD['bundle_manifest_sha256']
legacy.authorise(OWNER_RECORD,'upload_exploratory_b1_development_v1',{
 'archive_sha256':EXPECTED_DATA_SHA256,'bundle_manifest_sha256':EXPECTED_MANIFEST_SHA256},opt_in=True)
assert GPU_PREFLIGHT['status']=='PASS'
assert {n:legacy.sha256(root/n) for n in EXPECTED_FILES}==EXPECTED_FILES
print('Owner scope bound to verified development archive and unchanged code. No data uploaded yet.')


In [ ]:
# Upload exactly the733-development-source archive; verifier rejects extras/Test/holds/tampering.
assert GPU_PREFLIGHT['status']=='PASS' and CLOUD_DATA_AUTHORIZATION['cloud_upload_authorized'] is True
archive_path=pathlib.Path('/content')/EXPECTED_DATA_ZIP
data_root=pathlib.Path('/content/b1_pp1_pilot/development')
assert not archive_path.exists() and not data_root.exists(), 'STOP: do not overwrite or repeat upload/extraction'
uploaded_data=files.upload()
assert set(uploaded_data)=={EXPECTED_DATA_ZIP}, 'STOP: wrong file or more than one file'
archive_path.write_bytes(uploaded_data[EXPECTED_DATA_ZIP]); del uploaded_data
gc.collect()
PACKAGE_VERIFICATION=pilot.extract_verified_zip(archive_path,data_root,EXPECTED_DATA_SHA256,EXPECTED_MANIFEST_SHA256)
print(json.dumps(PACKAGE_VERIFICATION,indent=2))


In [ ]:
# Explicitly authorized official V2 download only; no alternative weights or install.
assert OWNER_RECORD['official_resnet50_v2_download_authorized'] is True
assert GPU_PREFLIGHT['status']=='PASS'
assert weights_source.WEIGHTS_URL=='https://download.pytorch.org/models/resnet50-11ad3fa6.pth'
assert torchvision.models.ResNet50_Weights.IMAGENET1K_V2.url==weights_source.WEIGHTS_URL
status=weights_source.pretrained_status(); checkpoint=pathlib.Path(status['expected_path'])
if status['status']!='documented_cache_present':
    assert not checkpoint.exists(), 'STOP: invalid existing checkpoint; no overwrite or fallback'
    checkpoint.parent.mkdir(parents=True,exist_ok=True)
    marker=checkpoint.parent/'b1_owner_v1_weight_download_attempt.json'
    with marker.open('x') as f: json.dump({'owner_date':OWNER_RECORD['date'],'official_url':weights_source.WEIGHTS_URL},f)
    torch.hub.download_url_to_file(weights_source.WEIGHTS_URL,str(checkpoint),hash_prefix='11ad3fa6',progress=True)
WEIGHT_RECORD=weights_source.pretrained_status()
assert WEIGHT_RECORD['status']=='documented_cache_present'
assert len(WEIGHT_RECORD['sha256'])==64 and WEIGHT_RECORD['sha256'].startswith('11ad3fa6')
print(json.dumps(WEIGHT_RECORD,indent=2))


In [ ]:
# Bind actual weights/code/protocol hashes only after verification; one bounded real run.
from datetime import datetime, timezone
assert OWNER_RECORD['training_authorized'] is True and GPU_PREFLIGHT['status']=='PASS'
assert {n:legacy.sha256(root/n) for n in EXPECTED_FILES}==EXPECTED_FILES
run_id='pp1_b1_mgi1to4_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output=pathlib.Path('/content/b1_pp1_pilot/results')/run_id
TRAINING_AUTHORIZATION={**OWNER_RECORD,'scope':pilot.SCOPE,
 'bundle_manifest_sha256':EXPECTED_MANIFEST_SHA256,'protocol_sha256':trainer.digest(pilot.protocol()),
 'code_sha256':trainer.digest(pilot.code_hashes()),'weights_sha256':WEIGHT_RECORD['sha256'],
 'output_run_id':run_id}
legacy.authorise(TRAINING_AUTHORIZATION,pilot.SCOPE,{
 'bundle_manifest_sha256':EXPECTED_MANIFEST_SHA256,'protocol_sha256':trainer.digest(pilot.protocol()),
 'code_sha256':trainer.digest(pilot.code_hashes()),'weights_sha256':WEIGHT_RECORD['sha256'],
 'output_run_id':run_id},opt_in=True)
control=pathlib.Path('/content/b1_pp1_pilot/owner_execution_v1')
control.mkdir(parents=True,exist_ok=False)
for name,value in [('owner_record.json',OWNER_RECORD),('training_authorization.json',TRAINING_AUTHORIZATION),
                   ('gpu_preflight.json',GPU_PREFLIGHT),('weights.json',WEIGHT_RECORD),
                   ('source_hashes_before.json',EXPECTED_FILES)]:
    (control/name).write_text(json.dumps(value,indent=2)+'\n')
RUN_PILOT=True
print(legacy.NOTICE,flush=True)
print('Starting ONE head-only run: maximum5epochs,patience2. Training output appears after completion; no automatic retry.',flush=True)
results=pilot.run_pilot(data_root,EXPECTED_MANIFEST_SHA256,TRAINING_AUTHORIZATION,output,opt_in=True)
assert {n:legacy.sha256(root/n) for n in EXPECTED_FILES}==EXPECTED_FILES
print(json.dumps(results,indent=2,allow_nan=False))


In [ ]:
# Download actual results, provenance and file hashes; do not claim success before this cell completes.
assert RUN_PILOT and (output/'pilot_results.json').is_file()
assert {n:legacy.sha256(root/n) for n in EXPECTED_FILES}==EXPECTED_FILES
for path in control.iterdir(): shutil.copyfile(path,output/path.name)
(output/'source_hashes_after.json').write_text(json.dumps(EXPECTED_FILES,indent=2)+'\n')
actual={str(p.relative_to(output)):legacy.sha256(p) for p in sorted(output.rglob('*')) if p.is_file()}
assert len(list((output/'gradcam').glob('*/*.png')))==9, 'STOP: require three CAM triplets'
(output/'results_file_hashes.json').write_text(json.dumps(actual,indent=2)+'\n')
result_zip=shutil.make_archive('/content/'+run_id+'_results','zip',output)
print('Actual result ZIP:',result_zip)
print('SHA256:',legacy.sha256(result_zip))
print(legacy.NOTICE)
files.download(result_zip)
